In [5]:
import numpy as np
import sys

scopepath = '/Users/sergivela/Documents/SCOPE/Program/'
sys.path.append(scopepath)

import Scope
from Scope import *

import matplotlib.pyplot as plt
from rdkit.Chem import Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import Descriptors, rdMolDescriptors
from sklearn.decomposition import PCA
from rdkit import Chem
import mols2grid

import yaml
import dscribe
from dscribe.descriptors import SOAP
from ase import Atoms

In [6]:
env_path = "/Users/sergivela/Documents/SCOPE/Program/environment_lemma.npy"
env = load_binary(env_path)

This code compute and saves SOAP 3 (see TSNE file) as variable

Compute_soap3 includes an additional SOAP descriptor for the coordination sphere alone (without the metal), computed as soap_n_coord_center.

NB: To see better use and implementation of SOAP (where we can modify the hyperparameters) see ML_Double_K_SOAP


In [8]:
systems = load_binary(env.sys_path+"unique_systems.npy")

In [15]:
def get_symbols_and_coords(molecule):
    symbols = []
    coords = []

    for at in molecule.atoms:
        symbols.append(at.label)      # element symbol
        coords.append(at.coord)       # [x, y, z] list

    return symbols, np.array(coords)

def get_symbols_and_coords_no_H(molecule):
    symbols, coords = get_symbols_and_coords(molecule)
    # Create mask to exclude H
    mask = [s != "H" for s in symbols]
    # Apply mask
    filtered_symbols = [s for s, keep in zip(symbols, mask) if keep]
    filtered_coords = [c for c, keep in zip(coords, mask) if keep]
    return filtered_symbols, filtered_coords

In [16]:
# Step 1: Collect all species used across your dataset
all_species = set()

for sys in systems:
    refcode = sys.HS_ref_mol.ligands[0].get_parent("cell").refcode
    #print('Molecule REFCODE:', refcode)
    
    if refcode == "NAPYFE10" or refcode == "WOJLUC" or refcode == "XUPDUH02":
        continue
    found, HS_state = find_state(sys.HS_ref_mol, "b3lyp_opt")
    mol = HS_state.get_moleclist()[0]
    symbols, coords = get_symbols_and_coords_no_H(mol)
    structure = Atoms(symbols=symbols, positions=coords)
    all_species.update(structure.get_chemical_symbols())

global_species = sorted(all_species)  # or use a manually defined order

soap = SOAP(
    species=global_species,
    r_cut=5.0,
    n_max=8,
    l_max=6,
    sigma=1.0,
    rbf="gto",
    sparse=False,
    periodic=False,
    average="outer"  # per-atom SOAP vectors
)


**SOAP 3**

In [17]:
def compute_soap3(mol):
    symbols, coords = get_symbols_and_coords_no_H(mol)
    # 1. Get full structure (without hydrogens)
    full_structure = Atoms(symbols=symbols, positions=coords)

    # 2. Get coordination sphere: Fe + coordinating atoms
    metal = mol.metals[0]  # SCOPE Metal object
    coordination_sphere = metal.get_coord_sphere()  # Returns list of atoms (not ASE)

    # Convert coordination sphere to ASE Atoms object
    cs_symbols = [atom.label for atom in coordination_sphere]
    cs_coords = [atom.coord for atom in coordination_sphere]
    cs_atoms = Atoms(symbols=cs_symbols, positions=cs_coords)

    # Convert metal to ASE Atoms object
    metal_symbol = metal.label
    metal_coord = metal.coord
    metal_atom = Atoms(symbols=[metal_symbol], positions=[metal_coord])

    # Combine them
    atoms_fe_n = cs_atoms + metal_atom

    # 3. Get positions of Fe-N atoms
    fe_n_positions = atoms_fe_n.get_positions()

    # 4. Match positions to indices in full_structure
    fe_n_indices = []
    for i, atom in enumerate(full_structure):
        for pos in fe_n_positions:
            if np.allclose(atom.position, pos, atol=0.1):  # Adjust tolerance if needed
                fe_n_indices.append(i)
                break

    # 5. Create Atoms objects for Fe-N and non-Fe-N parts
    atoms_fe_n_final = full_structure[fe_n_indices]
    atoms_non_fe_n = full_structure[[i for i in range(len(full_structure)) if i not in fe_n_indices]]

    # 6. Define SOAP species
    soap_fe_n = soap.create(atoms_fe_n_final)
    soap_non_fe_n = soap.create(atoms_non_fe_n)

    # 7. Compute SOAP for the coordination sphere centered around the metal (soap_n_coord_center)
    # The coordination sphere can be thought of as a cluster of atoms surrounding the metal
    soap_n_coord_center = soap.create(cs_atoms)  # SOAP descriptor for the coordination environment

    return soap_fe_n, soap_non_fe_n, soap_n_coord_center


In [18]:
for sys in systems:
    refcode = sys.HS_ref_mol.ligands[0].get_parent("cell").refcode

    if refcode in {"NAPYFE10", "WOJLUC", "XUPDUH02"}:
        continue

    # === HS state
    found, HS_state = find_state(sys.HS_ref_mol, "b3lyp_opt")
    mol1 = HS_state.get_moleclist()[0]
    desc_fe_n, desc_non_fe_n, desc_n_coord_center = compute_soap3(mol1)

    # Attach individual components to HS_ref_mol
    sys.HS_ref_mol.soap3_fe_n = desc_fe_n
    sys.HS_ref_mol.soap3_non_fe_n = desc_non_fe_n
    sys.HS_ref_mol.soap3_n_coord_center = desc_n_coord_center

    # === LS state
    found, LS_state = find_state(sys.LS_ref_mol, "b3lyp_opt")
    mol2 = LS_state.get_moleclist()[0]
    desc_fe_n2, desc_non_fe_n2, desc_n_coord_center2 = compute_soap3(mol2)

    # Attach individual components to LS_ref_mol
    sys.LS_ref_mol.soap3_fe_n = desc_fe_n2
    sys.LS_ref_mol.soap3_non_fe_n = desc_non_fe_n2
    sys.LS_ref_mol.soap3_n_coord_center = desc_n_coord_center2


In [19]:
save_binary(systems, "systems_with_soap3.npy")

In [ ]:
len(systems)

610

: 